# Large Dataset Preparation and Audit

This notebook prepares and investigates the larger painting dataset for the main colour-space experiments.

The dataset is examined before training to establish its structure, class distribution, image integrity, and potential duplication across data partitions.

The original small dataset and its experimental results are retained for comparison.

In [1]:
# Mount Google Drive

from google.colab import drive
drive.mount("/content/drive")

# Set project directory

%cd /content/drive/MyDrive/DeepLearning

import sys
sys.path.insert(0, "/content/drive/MyDrive/DeepLearning")

# Imports

from pathlib import Path
import zipfile
import os
import pandas as pd
from collections import Counter

from src import config, data

# Locate large dataset archive

LARGE_DATASET_ARCHIVE = (
    config.DATA_RAW_DIR / "paintings.imscc"
)

assert LARGE_DATASET_ARCHIVE.exists(), (
    f"Dataset archive not found: {LARGE_DATASET_ARCHIVE}"
)

print("Large dataset archive found!")
print(f"Location: {LARGE_DATASET_ARCHIVE}")
print(f"File size: {LARGE_DATASET_ARCHIVE.stat().st_size / (1024**2):.2f} MB")

Mounted at /content/drive
/content/drive/MyDrive/DeepLearning
Large dataset archive found!
Location: /content/drive/MyDrive/DeepLearning/data/raw/paintings.imscc
File size: 281.60 MB


In [2]:
# Inspect archive structure without extracting files

with zipfile.ZipFile(LARGE_DATASET_ARCHIVE, "r") as archive:

    archive_files = [
        name for name in archive.namelist()
        if not name.endswith("/")
    ]

    print(f"Total files in archive: {len(archive_files)}")

    print("\nFirst 40 files:")
    for filename in archive_files[:40]:
        print(filename)

Total files in archive: 2423

First 40 files:
__MACOSX/._paintingsHUGE
paintingsHUGE/.DS_Store
__MACOSX/paintingsHUGE/._.DS_Store
__MACOSX/paintingsHUGE/._test
__MACOSX/paintingsHUGE/._train
__MACOSX/paintingsHUGE/._validation
paintingsHUGE/test/.DS_Store
__MACOSX/paintingsHUGE/test/._.DS_Store
__MACOSX/paintingsHUGE/test/._monet
__MACOSX/paintingsHUGE/test/._picasso
__MACOSX/paintingsHUGE/test/._vangogh
paintingsHUGE/train/.DS_Store
__MACOSX/paintingsHUGE/train/._.DS_Store
__MACOSX/paintingsHUGE/train/._monet
__MACOSX/paintingsHUGE/train/._picasso
__MACOSX/paintingsHUGE/train/._vangogh
paintingsHUGE/validation/.DS_Store
__MACOSX/paintingsHUGE/validation/._.DS_Store
__MACOSX/paintingsHUGE/validation/._monet
__MACOSX/paintingsHUGE/validation/._picasso
__MACOSX/paintingsHUGE/validation/._vangogh
paintingsHUGE/test/monet/0fca0fa0b8f0d47203efccde2fe056a1--monet-garden-painting-techniques copy 3.jpg
__MACOSX/paintingsHUGE/test/monet/._0fca0fa0b8f0d47203efccde2fe056a1--monet-garden-painting-

### Initial Dataset Structure Audit

The archive is inspected to determine the number of images in each dataset partition and artist category.

macOS metadata files and hidden system files are excluded from the image counts.

The audit establishes the dataset distribution before extraction and further integrity checks.

In [3]:
# Count actual images in each partition and class

from collections import defaultdict
from pathlib import PurePosixPath

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

dataset_counts = defaultdict(int)
extension_counts = Counter()

metadata_files = 0
image_files = 0

with zipfile.ZipFile(LARGE_DATASET_ARCHIVE, "r") as archive:

    for filename in archive.namelist():

        path = PurePosixPath(filename)

        # Exclude macOS metadata
        if (
            "__MACOSX" in path.parts
            or path.name == ".DS_Store"
            or path.name.startswith("._")
        ):
            metadata_files += 1
            continue

        # Only inspect image files
        if path.suffix.lower() not in image_extensions:
            continue

        # Expected structure:
        # paintingsHUGE/split/class/image.jpg

        if len(path.parts) != 4:
            continue

        root, split, artist, image_name = path.parts

        if root != "paintingsHUGE":
            continue

        if split not in ["train", "validation", "test"]:
            continue

        if artist not in ["monet", "picasso", "vangogh"]:
            continue

        dataset_counts[(split, artist)] += 1
        extension_counts[path.suffix.lower()] += 1
        image_files += 1


# Display distribution

distribution_df = (
    pd.Series(dataset_counts)
    .unstack(fill_value=0)
    .reindex(
        index=["train", "validation", "test"],
        columns=["monet", "picasso", "vangogh"]
    )
)

distribution_df["total"] = distribution_df.sum(axis=1)

distribution_df.loc["total"] = distribution_df.sum(axis=0)

print("DATASET DISTRIBUTION")
display(distribution_df)

print("\nTOTAL IMAGE FILES:", image_files)
print("METADATA FILES EXCLUDED:", metadata_files)

print("\nIMAGE FORMATS:")
display(pd.DataFrame(
    extension_counts.items(),
    columns=["extension", "count"]
))

DATASET DISTRIBUTION


,monet,picasso,vangogh,total
train,280,280,280,840
validation,80,80,80,240
test,40,40,40,120
total,400,400,400,1200



TOTAL IMAGE FILES: 1200
METADATA FILES EXCLUDED: 1223

IMAGE FORMATS:


,extension,count
0,.jpg,1128
1,.jpeg,64
2,.png,8


### Dataset Extraction

The large dataset is extracted into a separate processed-data directory.

The extraction excludes macOS metadata files and preserves the original train, validation, and test partitions.

The original archive is retained as the raw source.

In [4]:
# Define separate directory for the large dataset

LARGE_DATASET_DIR = (
    config.DATA_PROCESSED_DIR / "paintingsHUGE"
)

LARGE_DATASET_DIR.mkdir(parents=True, exist_ok=True)


# Extract only actual dataset files

with zipfile.ZipFile(LARGE_DATASET_ARCHIVE, "r") as archive:

    for filename in archive.namelist():

        path = PurePosixPath(filename)

        # Exclude macOS metadata
        if (
            "__MACOSX" in path.parts
            or path.name == ".DS_Store"
            or path.name.startswith("._")
        ):
            continue

        # Extract only files belonging to paintingsHUGE
        if not filename.startswith("paintingsHUGE/"):
            continue

        if filename.endswith("/"):
            continue

        destination = LARGE_DATASET_DIR.joinpath(*path.parts[1:])
        destination.parent.mkdir(parents=True, exist_ok=True)

        with archive.open(filename) as source:
            with open(destination, "wb") as target:
                target.write(source.read())


print("Dataset extraction completed!")
print(f"Extracted to: {LARGE_DATASET_DIR}")

Dataset extraction completed!
Extracted to: /content/drive/MyDrive/DeepLearning/data/processed/paintingsHUGE


### Image Integrity and Duplicate Detection

This section evaluates the integrity of the extracted dataset.

All images are checked for readability, and SHA256 hashes are calculated to identify byte-identical files.

Duplicate detection is performed across all dataset partitions to identify potential data leakage between training, validation, and test data.

In [5]:
# Audit image integrity and exact duplicates

import hashlib
from PIL import Image, UnidentifiedImageError

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

image_records = []
invalid_images = []

for split in ["train", "validation", "test"]:

    for artist in config.CLASSES:

        artist_dir = LARGE_DATASET_DIR / split / artist

        for image_path in artist_dir.iterdir():

            if image_path.suffix.lower() not in image_extensions:
                continue

            try:
                # Verify image readability
                with Image.open(image_path) as img:
                    img.verify()

                # Calculate SHA256 hash
                sha256 = hashlib.sha256(
                    image_path.read_bytes()
                ).hexdigest()

                image_records.append({
                    "split": split,
                    "artist": artist,
                    "filename": image_path.name,
                    "path": str(image_path),
                    "sha256": sha256
                })

            except (UnidentifiedImageError, OSError, ValueError) as error:

                invalid_images.append({
                    "path": str(image_path),
                    "error": str(error)
                })


image_audit_df = pd.DataFrame(image_records)

print(f"Readable images: {len(image_audit_df)}")
print(f"Invalid images: {len(invalid_images)}")

# Identify exact duplicates

duplicate_mask = image_audit_df.duplicated(
    subset="sha256",
    keep=False
)

exact_duplicates_df = (
    image_audit_df[duplicate_mask]
    .sort_values("sha256")
)

print(f"\nImages involved in exact duplicates: {len(exact_duplicates_df)}")
print(f"Unique duplicate hashes: {exact_duplicates_df['sha256'].nunique()}")

if not exact_duplicates_df.empty:
    display(
        exact_duplicates_df[
            ["split", "artist", "filename", "sha256"]
        ]
    )

if invalid_images:
    display(pd.DataFrame(invalid_images))

Readable images: 1200
Invalid images: 0

Images involved in exact duplicates: 1200
Unique duplicate hashes: 299


,split,artist,filename,sha256
1175,test,vangogh,vincent-van-gogh-91991_960_720 copy.jpg,00e34c7cbc2626facd9dc5e9227897aad8bf46c30c87b8...
1186,test,vangogh,vincent-van-gogh-91991_960_720.jpg,00e34c7cbc2626facd9dc5e9227897aad8bf46c30c87b8...
1188,test,vangogh,vincent-van-gogh-91991_960_720 copy 2.jpg,00e34c7cbc2626facd9dc5e9227897aad8bf46c30c87b8...
1192,test,vangogh,vincent-van-gogh-91991_960_720 copy 3.jpg,00e34c7cbc2626facd9dc5e9227897aad8bf46c30c87b8...
125,train,monet,monet-paintings-wheatstacks-end-of-summer-1890...,01425120b37a9e10f9ea67a74a24928492726e6f4bf6d2...
...,...,...,...,...
233,train,monet,Bain-a-la-Grenouillere-1869-Claude-Monet copy.jpg,ff528e695eaede5fd35db937b3ff984dce6614e03a7d13...
1029,validation,vangogh,01.jpg,ffa037448c86d82edb2a80b5cae10a8354f0a73062eb25...
1057,validation,vangogh,01 copy 3.jpg,ffa037448c86d82edb2a80b5cae10a8354f0a73062eb25...
1038,validation,vangogh,01 copy.jpg,ffa037448c86d82edb2a80b5cae10a8354f0a73062eb25...


### Duplicate Distribution and Data Leakage Investigation

The initial integrity audit identified substantial exact duplication within the dataset.

This section investigates how duplicate images are distributed across dataset partitions and artist categories.

Particular attention is given to duplicates appearing across training, validation, and test partitions, as these may introduce data leakage and compromise the reliability of model evaluation.

In [6]:
# Group identical images by SHA256 hash

duplicate_groups = (
    image_audit_df
    .groupby("sha256")
    .agg(
        copies=("filename", "count"),
        splits=("split", lambda x: sorted(set(x))),
        artists=("artist", lambda x: sorted(set(x))),
        filenames=("filename", list),
        paths=("path", list)
    )
    .reset_index()
)

# Keep groups containing more than one copy

duplicate_groups = duplicate_groups[
    duplicate_groups["copies"] > 1
].copy()

# Identify cross-split and cross-class duplication

duplicate_groups["cross_split"] = (
    duplicate_groups["splits"].apply(len) > 1
)

duplicate_groups["cross_artist"] = (
    duplicate_groups["artists"].apply(len) > 1
)

print("DUPLICATE GROUP SUMMARY")
print(f"Unique image contents: {image_audit_df['sha256'].nunique()}")
print(f"Duplicate groups: {len(duplicate_groups)}")
print(f"Cross-split duplicate groups: {duplicate_groups['cross_split'].sum()}")
print(f"Cross-artist duplicate groups: {duplicate_groups['cross_artist'].sum()}")

print("\nDUPLICATES BY PARTITION")
display(
    image_audit_df.groupby("split").agg(
        total_files=("sha256", "size"),
        unique_images=("sha256", "nunique")
    )
)

print("\nCROSS-SPLIT DUPLICATES")
display(
    duplicate_groups.loc[
        duplicate_groups["cross_split"],
        ["copies", "splits", "artists", "filenames"]
    ].head(20)
)

print("\nCROSS-ARTIST DUPLICATES")
display(
    duplicate_groups.loc[
        duplicate_groups["cross_artist"],
        ["copies", "splits", "artists", "filenames"]
    ].head(20)
)

DUPLICATE GROUP SUMMARY
Unique image contents: 299
Duplicate groups: 299
Cross-split duplicate groups: 1
Cross-artist duplicate groups: 0

DUPLICATES BY PARTITION


,total_files,unique_images
split,,
test,120,30
train,840,210
validation,240,60



CROSS-SPLIT DUPLICATES


,copies,splits,artists,filenames
142,8,"[train, validation]",[picasso],"[hb_1990.192.jpg, hb_1990.192 copy.jpg, hb_199..."



CROSS-ARTIST DUPLICATES


,copies,splits,artists,filenames


In [7]:
# Compare unique images in the large dataset with the small dataset

small_dataset_root = data.get_dataset_root()

print("Small dataset root:", small_dataset_root)


def calculate_file_hash(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def collect_unique_image_hashes(dataset_root):

    records = []

    for split in ["train", "validation", "test"]:

        for artist in config.CLASSES:

            artist_dir = Path(dataset_root) / split / artist

            for image_path in artist_dir.iterdir():

                if image_path.suffix.lower() not in image_extensions:
                    continue

                records.append({
                    "split": split,
                    "artist": artist,
                    "filename": image_path.name,
                    "sha256": calculate_file_hash(image_path)
                })

    return pd.DataFrame(records)


small_images_df = collect_unique_image_hashes(
    small_dataset_root
)

large_unique_df = image_audit_df.drop_duplicates(
    subset="sha256"
).copy()


small_hashes = set(small_images_df["sha256"])
large_hashes = set(large_unique_df["sha256"])

shared_hashes = small_hashes.intersection(large_hashes)

print("\nDATASET OVERLAP")
print("----------------------------")
print(f"Unique small dataset images: {len(small_hashes)}")
print(f"Unique large dataset images: {len(large_hashes)}")
print(f"Shared image contents: {len(shared_hashes)}")
print(f"Only in small dataset: {len(small_hashes - large_hashes)}")
print(f"Only in large dataset: {len(large_hashes - small_hashes)}")

Small dataset root: /content/drive/MyDrive/DeepLearning/data/processed/paintingsSMALL/paintingsSMALL

DATASET OVERLAP
----------------------------
Unique small dataset images: 299
Unique large dataset images: 299
Shared image contents: 299
Only in small dataset: 0
Only in large dataset: 0
